# 03 Local Diffusion Refinement

This notebook takes the `seed_composite` and `refinement_mask` produced by face-DragGAN notebook `04`.

At this point:

- the replacement dog has already been pasted into the scene
- the geometry is roughly correct
- but the edges, seams, revealed background, and contact-shadow details are still not natural enough

This notebook is responsible only for **local refinement**, not for regenerating the full image from scratch.

Its design boundary is explicit:

1. preserve the geometry already established in `02`
2. run inpainting only in a narrow repair region
3. export a cleaner final result plus clear before/after visualizations


## Step 05 - Diffusion Refinement

This is the existing diffusion refinement stage, connected to the true-DragGAN face branch output from notebook `04`.

In [ ]:
%matplotlib inline

# Route name controls the output folder under data/outputs/face_draggan_pipeline/05_diffusion_refine/.
ROUTE_MODE = "single_model_refine"

# Choose exactly one inpainting model for this notebook run.
# Options: "realisticvision15_inpaint", "realvisxl_inpaint", "sdxl_inpaint".
INPAINT_MODEL_PRESET = "realvisxl_inpaint"

# Uminosachi/realisticVisionV51_v51VAE-inpainting. Current reliable local default.
REALISTICVISION15_STEPS = 100
REALISTICVISION15_MAX_GENERATION_SIDE = 768
REALISTICVISION15_GUIDANCE_SCALE = 6.5

# OzzyGT/RealVisXL_V4.0_inpainting. SDXL-family realistic backup.
REALVISXL_STEPS = 28
REALVISXL_MAX_GENERATION_SIDE = 1024
REALVISXL_GUIDANCE_SCALE = 6.5

# diffusers/stable-diffusion-xl-1.0-inpainting-0.1. Official SDXL inpaint baseline.
SDXL_INPAINT_STEPS = 28
SDXL_INPAINT_MAX_GENERATION_SIDE = 1024
SDXL_INPAINT_GUIDANCE_SCALE = 6.5

# Optional whole-scene downscale before diffusion. Keep 1.0 for quality.
SCENE_DOWNSCALE = 1.0

# Strong inpainting pass. Higher strength and generation side make SDXL visibly repair the masked region.
DIFFUSION_STRENGTH = 0.72
DIFFUSION_MASK_MODE = "residual_plus_outer"  # options: "residual_only", "residual_plus_outer", "none"

# Mask geometry controls.
RESIDUAL_HOLE_DILATE_KERNEL = 3
DOG_PROTECT_DILATE_KERNEL = 5

# Two-mask inpainting, matching the main pipeline:
# - generation_mask is wide, so the model sees enough context.
# - GENERATION_MASK_INCLUDE_DOG_EDGE controls whether that wide mask may include a thin dog-edge band.
#   Set False for complex backgrounds where including dog pixels confuses the inpaint model.
# - commit_mask is narrower and softly blended back, so we do not accept broad changes to the dog/body.
GENERATION_MASK_INCLUDE_DOG_EDGE = True
GENERATION_MASK_OUTER_DILATE_KERNEL = 65
GENERATION_MASK_DOG_EDGE_KERNEL = 13
DOG_CORE_PROTECT_KERNEL = 25
COMMIT_MASK_OUTER_DILATE_KERNEL = 11
COMMIT_MASK_BLUR = 21

# Legacy diagnostics retained for comparison plots.
DOG_EDGE_NO_DIFFUSE_KERNEL = 17
DOG_EDGE_SOFT_PRESERVE_KERNEL = 29
MASK_CLOSE_KERNEL = 3
PRE_DIFFUSION_BLEND_BLUR = 7
DIFFUSION_BLEND_BLUR = 3
LOCAL_DIFFUSION_ONLY = False


## Model Strategy: Small Local Inpainting For The Current 02 Batch

This notebook reads the pair list directly from face-DragGAN notebook `04` output manifests and applies the same local-inpainting setup to each discovered pair:

1. use `stable-diffusion-v1-5/stable-diffusion-inpainting`
2. keep the generation resolution small
3. keep the number of diffusion steps low

This stage is designed as a local repair tool, not a prompt-heavy scene generator.

References:

- https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-inpainting
- https://huggingface.co/docs/diffusers/main/api/pipelines/stable_diffusion/inpaint


In [ ]:
import json
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "outputs" / "face_draggan_pipeline" / "04_body_warp_after_true_draggan").exists() and (candidate / "notebooks").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


def print_pair_header(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a local CUDA GPU, but no CUDA device is available.")

PROJECT_ROOT = find_project_root()
WARP_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "04_body_warp_after_true_draggan"
REFINE_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "05_diffusion_refine"
ROUTE_CONFIG = {
    "mode": ROUTE_MODE,
    "selected_model_preset": INPAINT_MODEL_PRESET,
    "local_diffusion_only": bool(LOCAL_DIFFUSION_ONLY),
    "realisticvision15_steps": int(REALISTICVISION15_STEPS),
    "realisticvision15_max_side": int(REALISTICVISION15_MAX_GENERATION_SIDE),
    "realisticvision15_guidance": float(REALISTICVISION15_GUIDANCE_SCALE),
    "realvisxl_steps": int(REALVISXL_STEPS),
    "realvisxl_max_side": int(REALVISXL_MAX_GENERATION_SIDE),
    "realvisxl_guidance": float(REALVISXL_GUIDANCE_SCALE),
    "sdxl_inpaint_steps": int(SDXL_INPAINT_STEPS),
    "sdxl_inpaint_max_side": int(SDXL_INPAINT_MAX_GENERATION_SIDE),
    "sdxl_inpaint_guidance": float(SDXL_INPAINT_GUIDANCE_SCALE),
    "scene_downscale": float(SCENE_DOWNSCALE),
    "diffusion_strength": float(DIFFUSION_STRENGTH),
    "diffusion_mask_mode": DIFFUSION_MASK_MODE,
    "residual_hole_dilate_kernel": int(RESIDUAL_HOLE_DILATE_KERNEL),
    "dog_protect_dilate_kernel": int(DOG_PROTECT_DILATE_KERNEL),
    "generation_mask_include_dog_edge": bool(GENERATION_MASK_INCLUDE_DOG_EDGE),
    "generation_mask_outer_dilate_kernel": int(GENERATION_MASK_OUTER_DILATE_KERNEL),
    "generation_mask_dog_edge_kernel": int(GENERATION_MASK_DOG_EDGE_KERNEL),
    "dog_core_protect_kernel": int(DOG_CORE_PROTECT_KERNEL),
    "commit_mask_outer_dilate_kernel": int(COMMIT_MASK_OUTER_DILATE_KERNEL),
    "commit_mask_blur": int(COMMIT_MASK_BLUR),
    "dog_edge_no_diffuse_kernel": int(DOG_EDGE_NO_DIFFUSE_KERNEL),
    "dog_edge_soft_preserve_kernel": int(DOG_EDGE_SOFT_PRESERVE_KERNEL),
    "mask_close_kernel": int(MASK_CLOSE_KERNEL),
    "diffusion_blend_blur": int(DIFFUSION_BLEND_BLUR),
    "pre_diffusion_blend_blur": int(PRE_DIFFUSION_BLEND_BLUR),
    "description": "Run diffusion on the selected local repair mask; no automatic mask-size skip is applied.",
}
OUTPUT_ROOT = REFINE_ROOT / ROUTE_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ACTIVE_WARP_MANIFEST_PATH = WARP_ROOT / "_active_batch_manifest.json"
WARP_BATCH_MANIFEST_PATH = WARP_ROOT / "batch_manifest.json"


def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


PAIR_JOBS = []
active_manifest = load_json_if_exists(ACTIVE_WARP_MANIFEST_PATH)
batch_manifest = load_json_if_exists(WARP_BATCH_MANIFEST_PATH)

if active_manifest:
    manifest_rows = active_manifest
    ACTIVE_INPUT_MODE = "follow_notebook_02_active_output"
    MANIFEST_PATH_USED = ACTIVE_WARP_MANIFEST_PATH
elif batch_manifest:
    manifest_rows = batch_manifest
    ACTIVE_INPUT_MODE = "follow_notebook_02_batch_manifest"
    MANIFEST_PATH_USED = WARP_BATCH_MANIFEST_PATH
else:
    raise FileNotFoundError(
        "No warp manifest was found for notebook 05. Run face-DragGAN notebook 04 first so it can write "
        f"{ACTIVE_WARP_MANIFEST_PATH.name} or {WARP_BATCH_MANIFEST_PATH.name}."
    )

for idx, item in enumerate(manifest_rows, start=1):
    metadata_path = Path(item["metadata_path"])
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Warp metadata listed in the manifest does not exist: {metadata_path}. "
            "Re-run face-DragGAN notebook 04 to refresh the warp package."
        )
    pair_name = item["pair_name"]
    PAIR_JOBS.append(
        {
            "pair_id": item.get("pair_id", f"pair_{idx:02d}"),
            "pair_name": pair_name,
            "metadata_path": metadata_path,
            "output_dir": OUTPUT_ROOT / pair_name,
        }
    )

DEFAULT_POSITIVE_PROMPT = "photorealistic local background repair, seamless wall pavement grass, preserve the pasted dog unchanged"
DEFAULT_NEGATIVE_PROMPT = "extra dog, duplicate dog, dog fragments, extra limbs, extra head, changed dog body, wrong anatomy, blurry halo, gray outline, cartoon, text, watermark"

MODEL_PRESETS = {
    "realisticvision15_inpaint": {
        "model_id": "Uminosachi/realisticVisionV51_v51VAE-inpainting",
        "pipeline_family": "auto_inpaint",
        "variant": None,
        "max_side": int(REALISTICVISION15_MAX_GENERATION_SIDE),
        "steps": int(REALISTICVISION15_STEPS),
        "guidance": float(REALISTICVISION15_GUIDANCE_SCALE),
    },
    "realvisxl_inpaint": {
        "model_id": "OzzyGT/RealVisXL_V4.0_inpainting",
        "pipeline_family": "auto_inpaint",
        "variant": "fp16",
        "max_side": int(REALVISXL_MAX_GENERATION_SIDE),
        "steps": int(REALVISXL_STEPS),
        "guidance": float(REALVISXL_GUIDANCE_SCALE),
    },
    "sdxl_inpaint": {
        "model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        "pipeline_family": "auto_inpaint",
        "variant": "fp16",
        "max_side": int(SDXL_INPAINT_MAX_GENERATION_SIDE),
        "steps": int(SDXL_INPAINT_STEPS),
        "guidance": float(SDXL_INPAINT_GUIDANCE_SCALE),
    },
}

if INPAINT_MODEL_PRESET not in MODEL_PRESETS:
    raise KeyError(f"Unknown INPAINT_MODEL_PRESET: {INPAINT_MODEL_PRESET}. Available presets: {sorted(MODEL_PRESETS)}")

ACTIVE_MODEL_CONFIG = MODEL_PRESETS[INPAINT_MODEL_PRESET]
ACTIVE_DIFFUSION_STAGE = {
    "model_preset": INPAINT_MODEL_PRESET,
    "model_id": ACTIVE_MODEL_CONFIG["model_id"],
    "steps": int(ACTIVE_MODEL_CONFIG["steps"]),
    "max_side": int(ACTIVE_MODEL_CONFIG["max_side"]),
    "guidance": float(ACTIVE_MODEL_CONFIG["guidance"]),
}

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
SEED = int(os.environ.get("DOG_REFINER_SEED", "7"))

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Route mode:", ROUTE_MODE)
print("Route config:", ROUTE_CONFIG)
print("Selected inpaint model:", ACTIVE_DIFFUSION_STAGE)
print("Diffusion mask mode:", DIFFUSION_MASK_MODE)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Input mode:", ACTIVE_INPUT_MODE)
print("Manifest used:", MANIFEST_PATH_USED)
print("Discovered pair jobs:")
for job in PAIR_JOBS:
    print(" -", job["pair_id"], "|", job["pair_name"])


## Load The Seed Composite And The Repair Masks From 02

This notebook now reads two related masks from face-DragGAN notebook `04`:

- `refinement_mask`: the general local-repair mask
- `residual_hole_mask`: the precise region where the pasted replacement dog still leaves part of the original removed-dog hole uncovered

The residual-hole mask is especially useful for the blurry transition problem. Instead of enlarging the entire refinement mask, we only dilate this residual-hole region a little and merge it back into the final inpainting mask.


In [ ]:
def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_mask(path):
    return np.array(Image.open(path).convert("L")) > 127


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(path)


def save_mask(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def ensure_uint8(image):
    return np.clip(image, 0, 255).astype(np.uint8)


def plot_images(items, cols=3, figsize=(16, 10)):
    if not items:
        return
    rows = int(math.ceil(len(items) / cols))
    plt.figure(figsize=figsize)
    for idx, (title, image) in enumerate(items, start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap="gray")
        else:
            plt.imshow(image)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## Diffusion Helpers: Small-Model Inpainting For Local Repair

The implementation is intentionally simple:

- load a lighter inpainting pipeline
- resize to a practical local generation resolution
- edit only inside the repair mask
- blend the repaired patch back into the seed composite

This stage is meant to behave like a local repair tool, not a full scene re-generator.


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True


def resize_for_generation(image, mask, max_side=1024, multiple=16):
    h, w = image.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    new_w = max(multiple, int(round(w * scale / multiple)) * multiple)
    new_h = max(multiple, int(round(h * scale / multiple)) * multiple)
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return resized_image, resized_mask, scale


def downscale_scene_bundle(image, mask_map, scale, multiple=8):
    if scale >= 0.999:
        return image.copy(), {name: mask.copy() for name, mask in mask_map.items()}, 1.0
    h, w = image.shape[:2]
    new_w = max(multiple, int(round(w * scale / multiple)) * multiple)
    new_h = max(multiple, int(round(h * scale / multiple)) * multiple)
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_masks = {
        name: cv2.resize(mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        for name, mask in mask_map.items()
    }
    return resized_image, resized_masks, float(new_w / max(1, w))


def restore_to_canvas(image, target_shape):
    target_h, target_w = target_shape[:2]
    if image.shape[:2] == (target_h, target_w):
        return image
    return cv2.resize(image, (target_w, target_h), interpolation=cv2.INTER_CUBIC)


def masked_preview(image, mask):
    preview = image.copy()
    preview[mask] = [255, 255, 255]
    return preview


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def crop_around_mask(image, mask, margin=64):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return image
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)
    return image[y1:y2, x1:x2]


def soft_blend(original, generated, mask, blur_kernel=11):
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    soft = cv2.GaussianBlur(mask.astype(np.float32), (blur_kernel, blur_kernel), 0)[..., None]
    return ensure_uint8(generated.astype(np.float32) * soft + original.astype(np.float32) * (1.0 - soft))


def soft_blend_with_float_alpha(original, generated, alpha):
    alpha = np.clip(alpha.astype(np.float32), 0.0, 1.0)[..., None]
    return ensure_uint8(generated.astype(np.float32) * alpha + original.astype(np.float32) * (1.0 - alpha))


def make_soft_commit_alpha(commit_mask, blur_kernel):
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    return np.clip(cv2.GaussianBlur(commit_mask.astype(np.float32), (blur_kernel, blur_kernel), 0), 0.0, 1.0)


def load_inpaint_pipeline(model_id, model_config):
    from diffusers import AutoPipelineForInpainting

    kwargs = {
        "torch_dtype": torch.float16,
        "use_safetensors": True,
        "safety_checker": None,
        "requires_safety_checker": False,
    }
    variant = model_config.get("variant")
    if variant:
        kwargs["variant"] = variant
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    if LOCAL_DIFFUSION_ONLY:
        kwargs["local_files_only"] = True
    pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kwargs)
    pipe.enable_model_cpu_offload()
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    return pipe


def get_inpaint_pipeline(model_preset):
    model_config = MODEL_PRESETS[model_preset]
    model_id = model_config["model_id"]
    cached = globals().get("INPAINT_PIPELINE_CACHE")
    if cached is not None:
        cached_pipe, cached_preset, cached_model_id = cached
        if cached_preset == model_preset and cached_model_id == model_id:
            return cached
        globals().pop("INPAINT_PIPELINE_CACHE", None)
        try:
            del cached_pipe
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pipe = load_inpaint_pipeline(model_id, model_config)
    globals()["INPAINT_PIPELINE_CACHE"] = (pipe, model_preset, model_id)
    return globals()["INPAINT_PIPELINE_CACHE"]


def run_refinement(
    image,
    generation_mask,
    commit_mask,
    prompt,
    negative_prompt,
    model_preset,
    steps,
    guidance_scale,
    seed=7,
    max_side=512,
    stage_name="stage",
):
    if int(steps) <= 0:
        return image.copy(), f"{stage_name}_skipped", "none"
    if generation_mask.sum() < 10 or commit_mask.sum() < 10:
        return image.copy(), f"{stage_name}_no_edit_needed", "none"

    print(
        f"Running diffusion stage '{stage_name}': preset={model_preset}, steps={int(steps)}, "
        f"generation_pixels={int(generation_mask.sum())}, commit_pixels={int(commit_mask.sum())}"
    )
    pipe, pipe_kind, model_id = get_inpaint_pipeline(model_preset)
    gen_image, gen_mask, scale = resize_for_generation(image, generation_mask, max_side=max_side, multiple=8)
    generator = torch.Generator(device="cpu").manual_seed(seed)

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=Image.fromarray(gen_image),
        mask_image=Image.fromarray((gen_mask.astype(np.uint8) * 255)).convert("L"),
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(steps),
        strength=float(DIFFUSION_STRENGTH),
        generator=generator,
    ).images[0].convert("RGB")

    result_np = np.array(result)
    if result_np.shape[:2] != image.shape[:2]:
        result_np = cv2.resize(result_np, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_CUBIC)

    commit_alpha = make_soft_commit_alpha(commit_mask, COMMIT_MASK_BLUR)
    return soft_blend_with_float_alpha(image, result_np, commit_alpha), pipe_kind, model_id


def run_selected_refinement(image, generation_mask, commit_mask, prompt, negative_prompt, stage, seed=7):
    before_stage = image.copy()
    refined, pipe_kind, model_id = run_refinement(
        image,
        generation_mask,
        commit_mask,
        prompt=prompt,
        negative_prompt=negative_prompt,
        model_preset=stage["model_preset"],
        steps=stage["steps"],
        guidance_scale=stage["guidance"],
        seed=seed,
        max_side=stage["max_side"],
        stage_name=stage["model_preset"],
    )
    pixel_diff = np.abs(refined.astype(np.int16) - before_stage.astype(np.int16))
    record = {
        "model_preset": stage["model_preset"],
        "model_id": model_id,
        "pipeline_kind": pipe_kind,
        "steps": int(stage["steps"]),
        "max_side": int(stage["max_side"]),
        "guidance": float(stage["guidance"]),
        "seed": int(seed),
        "mean_abs_pixel_delta": round(float(pixel_diff.mean()), 6),
        "changed_pixels": int((pixel_diff.sum(axis=2) > 0).sum()),
    }
    return refined, record



def luminance_harmonize_from_reference(seed_composite, warped_mask, reference_image, reference_mask, strength=0.35):
    if warped_mask.sum() < 20 or reference_mask.sum() < 20:
        return seed_composite.copy()
    ref_lab = cv2.cvtColor(reference_image, cv2.COLOR_RGB2LAB).astype(np.float32)
    gen_lab = cv2.cvtColor(seed_composite, cv2.COLOR_RGB2LAB).astype(np.float32)
    ref_L = ref_lab[..., 0][reference_mask]
    gen_L = gen_lab[..., 0][warped_mask]
    ref_mean, ref_std = float(ref_L.mean()), float(ref_L.std() + 1e-4)
    gen_mean, gen_std = float(gen_L.mean()), float(gen_L.std() + 1e-4)
    adjusted = gen_lab[..., 0].copy()
    adjusted_vals = ((gen_L - gen_mean) / gen_std) * ref_std + ref_mean
    adjusted_vals = np.clip(adjusted_vals, 0.0, 255.0)
    blended_vals = (1.0 - strength) * gen_L + strength * adjusted_vals
    adjusted[warped_mask] = blended_vals
    gen_lab[..., 0] = adjusted
    return cv2.cvtColor(np.clip(gen_lab, 0.0, 255.0).astype(np.uint8), cv2.COLOR_LAB2RGB)


def build_contact_shadow_alpha(mask, shift_y=8, blur_kernel=31, strength=0.16):
    mask_u8 = mask.astype(np.uint8)
    base = cv2.erode(mask_u8, np.ones((5, 5), np.uint8), iterations=1)
    shifted = np.zeros_like(base)
    if shift_y < base.shape[0]:
        shifted[shift_y:] = base[:-shift_y]
    shadow = cv2.GaussianBlur(shifted.astype(np.float32), (blur_kernel, blur_kernel), 0)
    shadow = np.clip(shadow * float(strength), 0.0, 0.35)
    return shadow


def apply_contact_shadow(image, shadow_alpha):
    shaded = image.astype(np.float32).copy()
    shaded *= (1.0 - shadow_alpha[..., None])
    return ensure_uint8(shaded)


## Run Local Inpainting Refinement For Every Pair

The notebook now loops over every discovered pair, builds the final repair mask, runs local inpainting, saves the final outputs, and shows all visualizations for each pair.


In [ ]:
REFINE_RESULTS = []

for pair_index, job in enumerate(PAIR_JOBS, start=1):
    with open(job["metadata_path"], "r", encoding="utf-8") as f:
        warp_meta = json.load(f)
    with open(warp_meta["cutout_metadata_path"], "r", encoding="utf-8") as f:
        cutout_meta = json.load(f)

    seed_background = read_rgb(Path(warp_meta["outputs"]["seed_background"]))
    seed_composite_path = Path(warp_meta["outputs"].get("resolution_harmonized_seed", warp_meta["outputs"]["seed_composite"]))
    seed_composite = read_rgb(seed_composite_path)
    baseline_composite = read_rgb(Path(warp_meta["outputs"]["baseline_composite"]))
    warped_mask = read_mask(Path(warp_meta["outputs"]["warped_mask"]))
    refinement_mask = read_mask(Path(warp_meta["outputs"]["refinement_mask"]))
    residual_hole_mask = read_mask(Path(warp_meta["outputs"]["residual_hole_mask"])) if "residual_hole_mask" in warp_meta["outputs"] else np.zeros_like(refinement_mask)
    seam_ring_mask = read_mask(Path(warp_meta["outputs"]["seam_ring_mask"])) if "seam_ring_mask" in warp_meta["outputs"] else np.zeros_like(refinement_mask)
    original_image = read_rgb(Path(warp_meta["cutout_metadata_path"]).parent / "original_image.png")
    original_mask = read_mask(PROJECT_ROOT / cutout_meta["original"]["mask_path"])

    residual_kernel = int(RESIDUAL_HOLE_DILATE_KERNEL)
    if residual_kernel % 2 == 0:
        residual_kernel += 1
    protect_kernel = int(DOG_PROTECT_DILATE_KERNEL)
    if protect_kernel % 2 == 0:
        protect_kernel += 1
    close_kernel = int(MASK_CLOSE_KERNEL)
    if close_kernel % 2 == 0:
        close_kernel += 1
    generation_outer_kernel = int(GENERATION_MASK_OUTER_DILATE_KERNEL)
    if generation_outer_kernel % 2 == 0:
        generation_outer_kernel += 1
    generation_dog_edge_kernel = int(GENERATION_MASK_DOG_EDGE_KERNEL)
    if generation_dog_edge_kernel % 2 == 0:
        generation_dog_edge_kernel += 1
    dog_core_protect_kernel = int(DOG_CORE_PROTECT_KERNEL)
    if dog_core_protect_kernel % 2 == 0:
        dog_core_protect_kernel += 1
    commit_outer_kernel = int(COMMIT_MASK_OUTER_DILATE_KERNEL)
    if commit_outer_kernel % 2 == 0:
        commit_outer_kernel += 1
    edge_no_diffuse_kernel = int(DOG_EDGE_NO_DIFFUSE_KERNEL)
    if edge_no_diffuse_kernel % 2 == 0:
        edge_no_diffuse_kernel += 1
    edge_soft_preserve_kernel = int(DOG_EDGE_SOFT_PRESERVE_KERNEL)
    if edge_soft_preserve_kernel % 2 == 0:
        edge_soft_preserve_kernel += 1

    residual_hole_expanded = cv2.dilate(
        residual_hole_mask.astype(np.uint8),
        np.ones((residual_kernel, residual_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    protected_dog_zone = cv2.dilate(
        warped_mask.astype(np.uint8),
        np.ones((protect_kernel, protect_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    dog_edge_no_diffuse_zone = cv2.dilate(
        warped_mask.astype(np.uint8),
        np.ones((edge_no_diffuse_kernel, edge_no_diffuse_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    dog_edge_soft_preserve_zone = cv2.dilate(
        warped_mask.astype(np.uint8),
        np.ones((edge_soft_preserve_kernel, edge_soft_preserve_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    dog_edge_soft_preserve_zone = np.logical_and(dog_edge_soft_preserve_zone, ~dog_edge_no_diffuse_zone)

    outer_seam_only_mask = np.logical_and(seam_ring_mask, ~protected_dog_zone)
    outer_seam_for_diffusion = np.logical_and(outer_seam_only_mask, ~dog_edge_no_diffuse_zone)
    hole_fill_mask = np.logical_and(residual_hole_expanded, ~protected_dog_zone)
    hole_fill_for_diffusion = np.logical_or(
        np.logical_and(hole_fill_mask, ~dog_edge_no_diffuse_zone),
        np.logical_and(hole_fill_mask, residual_hole_mask),
    )

    if DIFFUSION_MASK_MODE == "none":
        final_refinement_mask = np.zeros_like(hole_fill_mask, dtype=bool)
    elif DIFFUSION_MASK_MODE == "residual_plus_outer":
        final_refinement_mask = np.logical_or(hole_fill_for_diffusion, outer_seam_for_diffusion)
    elif DIFFUSION_MASK_MODE == "residual_only":
        final_refinement_mask = hole_fill_for_diffusion.copy()
    else:
        raise ValueError(f"Unknown DIFFUSION_MASK_MODE: {DIFFUSION_MASK_MODE}")

    final_refinement_mask = cv2.morphologyEx(
        final_refinement_mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        np.ones((close_kernel, close_kernel), np.uint8),
    ).astype(bool)

    commit_mask = cv2.dilate(
        final_refinement_mask.astype(np.uint8),
        np.ones((commit_outer_kernel, commit_outer_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    commit_mask = np.logical_and(commit_mask, ~warped_mask)

    dog_core_protect_mask = cv2.erode(
        warped_mask.astype(np.uint8),
        np.ones((dog_core_protect_kernel, dog_core_protect_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    dog_edge_generation_band = np.logical_and(warped_mask, ~dog_core_protect_mask)
    dog_edge_generation_band = cv2.dilate(
        dog_edge_generation_band.astype(np.uint8),
        np.ones((generation_dog_edge_kernel, generation_dog_edge_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    generation_background_context = cv2.dilate(
        final_refinement_mask.astype(np.uint8),
        np.ones((generation_outer_kernel, generation_outer_kernel), np.uint8),
        iterations=1,
    ).astype(bool)
    if GENERATION_MASK_INCLUDE_DOG_EDGE:
        generation_mask = np.logical_or(generation_background_context, dog_edge_generation_band)
        generation_mask = np.logical_and(generation_mask, ~dog_core_protect_mask)
    else:
        generation_mask = np.logical_and(generation_background_context, ~warped_mask)
    generation_mask = cv2.morphologyEx(
        generation_mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        np.ones((close_kernel, close_kernel), np.uint8),
    ).astype(bool)

    luminance_harmonized_seed = luminance_harmonize_from_reference(seed_composite, warped_mask, original_image, original_mask, strength=0.35)
    contact_shadow_alpha = build_contact_shadow_alpha(warped_mask, shift_y=8, blur_kernel=31, strength=0.16)
    shadowed_seed = apply_contact_shadow(luminance_harmonized_seed, contact_shadow_alpha)
    contact_shadow_region = contact_shadow_alpha > 0.015
    pre_diffusion_seed = seed_composite.copy()
    pre_diffusion_seed[warped_mask] = luminance_harmonized_seed[warped_mask]
    pre_blend_mask = np.logical_or(hole_fill_mask, contact_shadow_region)
    if DIFFUSION_MASK_MODE == "residual_plus_outer":
        pre_blend_mask = np.logical_or(pre_blend_mask, outer_seam_only_mask)
    pre_diffusion_seed = soft_blend(
        pre_diffusion_seed,
        shadowed_seed,
        pre_blend_mask,
        blur_kernel=PRE_DIFFUSION_BLEND_BLUR,
    )

    print_pair_header(f"[05] Pair {pair_index}/{len(PAIR_JOBS)} :: {job['pair_name']}")
    print("Seed composite shape:", seed_composite.shape)
    print("Base refinement mask pixels:", int(refinement_mask.sum()))
    print("Residual hole mask pixels:", int(residual_hole_mask.sum()))
    print("Expanded residual hole mask pixels:", int(residual_hole_expanded.sum()))
    print("Outer seam-only mask pixels:", int(outer_seam_only_mask.sum()))
    print("Protected dog zone pixels:", int(protected_dog_zone.sum()))
    print("Dog edge no-diffuse zone pixels:", int(dog_edge_no_diffuse_zone.sum()))
    print("Dog edge soft-preserve zone pixels:", int(dog_edge_soft_preserve_zone.sum()))
    requested_mask_ratio = float(final_refinement_mask.mean())
    print("Final refinement mask pixels:", int(final_refinement_mask.sum()))
    print("Final refinement mask ratio:", round(requested_mask_ratio, 5))
    print("Generation mask pixels:", int(generation_mask.sum()))
    print("Commit mask pixels:", int(commit_mask.sum()))
    print("Generation mask includes dog edge:", bool(GENERATION_MASK_INCLUDE_DOG_EDGE))
    print("Dog core protected pixels:", int(dog_core_protect_mask.sum()))
    print("Dog edge generation-band pixels:", int(dog_edge_generation_band.sum()))

    plot_images(
        [
            ("Original image", original_image),
            ("Baseline composite", baseline_composite),
            ("Pose-guided seed composite", seed_composite),
            ("Luminance-harmonized seed", luminance_harmonized_seed),
            ("Contact shadow alpha", contact_shadow_alpha),
            ("Pre-diffusion seed", pre_diffusion_seed),
            ("Warped dog mask", warped_mask),
            ("Residual hole mask", residual_hole_mask),
            ("Outer seam-only mask", outer_seam_only_mask),
            ("Protected dog zone", protected_dog_zone),
            ("Dog edge no-diffuse zone", dog_edge_no_diffuse_zone),
            ("Dog edge soft-preserve zone", dog_edge_soft_preserve_zone),
            ("Base commit candidate mask", final_refinement_mask),
            ("Generation mask used by diffusion", generation_mask),
            ("Commit mask accepted back", commit_mask),
            ("Dog core protected from generation", dog_core_protect_mask),
            ("Dog edge allowed as generation context", dog_edge_generation_band),
        ],
        cols=3,
        figsize=(18, 16),
    )

    requested_refinement_mask = final_refinement_mask.copy()
    effective_refinement_mask = commit_mask.copy()
    effective_generation_mask = generation_mask.copy()
    print("Diffusion will run with the selected inpaint model if generation/commit masks are non-empty.")

    route_seed = pre_diffusion_seed
    route_masks = {
        "generation": effective_generation_mask,
        "commit": effective_refinement_mask,
        "requested_refinement": requested_refinement_mask,
        "base": refinement_mask,
        "outer": outer_seam_only_mask,
    }
    route_scene_scale = 1.0
    if SCENE_DOWNSCALE < 0.999:
        route_seed, route_masks, route_scene_scale = downscale_scene_bundle(pre_diffusion_seed, route_masks, SCENE_DOWNSCALE, multiple=8)
    route_generation_mask = route_masks["generation"]
    route_refinement_mask = route_masks["commit"]
    route_requested_refinement_mask = route_masks["requested_refinement"]
    route_base_mask = route_masks["base"]
    route_outer_mask = route_masks["outer"]
    preview_image = masked_preview(route_seed, route_generation_mask)

    print("Positive prompt:")
    print(DEFAULT_POSITIVE_PROMPT)
    print("\nNegative prompt:")
    print(DEFAULT_NEGATIVE_PROMPT)

    plot_images(
        [
            ("Seed composite", seed_composite),
            ("Pre-diffusion seed", pre_diffusion_seed),
            ("Route seed for diffusion", route_seed),
            ("Masked preview for diffusion", preview_image),
            ("Base refinement mask", route_base_mask),
            ("Expanded residual hole mask", residual_hole_expanded),
            ("Outer seam-only mask", route_outer_mask),
            ("Outer seam allowed for diffusion", outer_seam_for_diffusion),
            ("Requested refinement mask", route_requested_refinement_mask),
            ("Generation mask sent to diffusion", route_generation_mask),
            ("Commit mask accepted back", route_refinement_mask),
        ],
        cols=3,
        figsize=(18, 10),
    )

    refined_route_image, refinement_record = run_selected_refinement(
        route_seed,
        route_generation_mask,
        route_refinement_mask,
        prompt=DEFAULT_POSITIVE_PROMPT,
        negative_prompt=DEFAULT_NEGATIVE_PROMPT,
        stage=ACTIVE_DIFFUSION_STAGE,
        seed=SEED,
    )
    used_pipe_kind = refinement_record["pipeline_kind"]
    used_model_id = refinement_record["model_id"]
    refined_image = restore_to_canvas(refined_route_image, pre_diffusion_seed.shape)
    if route_refinement_mask.shape == final_refinement_mask.shape:
        route_refinement_mask_full = route_refinement_mask
    else:
        route_refinement_mask_full = cv2.resize(
            route_refinement_mask.astype(np.uint8),
            (final_refinement_mask.shape[1], final_refinement_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        ).astype(bool)
    if route_generation_mask.shape == final_refinement_mask.shape:
        route_generation_mask_full = route_generation_mask
    else:
        route_generation_mask_full = cv2.resize(
            route_generation_mask.astype(np.uint8),
            (final_refinement_mask.shape[1], final_refinement_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        ).astype(bool)

    print("Used pipeline kind:", used_pipe_kind)
    print("Used model id:", used_model_id)
    print("Refinement record:", refinement_record)

    if isinstance(route_refinement_mask_full, np.ndarray) and route_refinement_mask_full.ndim == 3:
        route_refinement_mask_full = route_refinement_mask_full[..., 0] > 127
    zoom_seed = crop_around_mask(pre_diffusion_seed, final_refinement_mask)
    zoom_refined = crop_around_mask(refined_image, final_refinement_mask)
    zoom_mask = crop_around_mask((route_refinement_mask_full.astype(np.uint8) * 255), final_refinement_mask)
    zoom_generation_mask = crop_around_mask((route_generation_mask_full.astype(np.uint8) * 255), final_refinement_mask)

    plot_images(
        [
            ("Baseline composite", baseline_composite),
            ("Pre-diffusion seed", pre_diffusion_seed),
            ("Refined result", refined_image),
            ("Zoom: seed", zoom_seed),
            ("Zoom: refined", zoom_refined),
            ("Zoom: generation mask", zoom_generation_mask),
            ("Zoom: commit mask", zoom_mask),
        ],
        cols=3,
        figsize=(18, 12),
    )

    output_dir = job["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)
    save_rgb(output_dir / "seed_composite.png", seed_composite)
    save_rgb(output_dir / "luminance_harmonized_seed.png", luminance_harmonized_seed)
    save_rgb(output_dir / "pre_diffusion_seed.png", pre_diffusion_seed)
    save_rgb(output_dir / "route_seed.png", route_seed)
    save_rgb(output_dir / "masked_preview.png", preview_image)
    save_mask(output_dir / "refinement_mask.png", requested_refinement_mask)
    save_mask(output_dir / "route_generation_mask.png", route_generation_mask)
    save_mask(output_dir / "route_refinement_mask.png", route_refinement_mask)
    save_mask(output_dir / "effective_generation_mask.png", effective_generation_mask)
    save_mask(output_dir / "effective_refinement_mask.png", effective_refinement_mask)
    save_mask(output_dir / "generation_mask.png", generation_mask)
    save_mask(output_dir / "commit_mask.png", commit_mask)
    save_mask(output_dir / "dog_core_protect_mask.png", dog_core_protect_mask)
    save_mask(output_dir / "dog_edge_generation_band.png", dog_edge_generation_band)
    save_mask(output_dir / "base_refinement_mask.png", refinement_mask)
    save_mask(output_dir / "residual_hole_mask.png", residual_hole_mask)
    save_mask(output_dir / "expanded_residual_hole_mask.png", residual_hole_expanded)
    save_mask(output_dir / "outer_seam_only_mask.png", outer_seam_only_mask)
    save_mask(output_dir / "protected_dog_zone.png", protected_dog_zone)
    save_mask(output_dir / "dog_edge_no_diffuse_zone.png", dog_edge_no_diffuse_zone)
    save_mask(output_dir / "dog_edge_soft_preserve_zone.png", dog_edge_soft_preserve_zone)
    save_mask(output_dir / "outer_seam_for_diffusion.png", outer_seam_for_diffusion)
    save_mask(output_dir / "pre_diffusion_blend_mask.png", pre_blend_mask)
    Image.fromarray(np.clip(contact_shadow_alpha * 255.0, 0, 255).astype(np.uint8)).save(output_dir / "contact_shadow_alpha.png")
    save_rgb(output_dir / "refined_final.png", refined_image)
    save_rgb(output_dir / "zoom_seed.png", zoom_seed)
    save_rgb(output_dir / "zoom_refined.png", zoom_refined)

    run_config = {
        "pair_name": job["pair_name"],
        "pair_index": pair_index,
        "warp_metadata_path": str(job["metadata_path"]),
        "route_mode": ROUTE_MODE,
        "route_config": ROUTE_CONFIG,
        "scene_downscale": SCENE_DOWNSCALE,
        "route_scene_scale": route_scene_scale,
        "selected_model_preset": INPAINT_MODEL_PRESET,
        "selected_model_config": ACTIVE_DIFFUSION_STAGE,
        "refinement_record": refinement_record,
        "local_diffusion_only": bool(LOCAL_DIFFUSION_ONLY),
        "used_pipeline_kind": used_pipe_kind,
        "used_model_id": used_model_id,
        "seed": SEED,
        "diffusion_strength": float(DIFFUSION_STRENGTH),
        "diffusion_mask_mode": DIFFUSION_MASK_MODE,
        "requested_refinement_mask_pixels": int(requested_refinement_mask.sum()),
        "effective_refinement_mask_pixels": int(effective_refinement_mask.sum()),
        "residual_hole_dilate_kernel": residual_kernel,
        "protect_dilate_kernel": protect_kernel,
        "generation_mask_pixels": int(generation_mask.sum()),
        "commit_mask_pixels": int(commit_mask.sum()),
        "requested_refinement_mask_ratio": float(requested_refinement_mask.mean()),
        "effective_refinement_mask_ratio": float(effective_refinement_mask.mean()),
        "generation_mask_ratio": float(generation_mask.mean()),
        "commit_mask_ratio": float(commit_mask.mean()),
        "generation_mask_include_dog_edge": bool(GENERATION_MASK_INCLUDE_DOG_EDGE),
        "generation_mask_outer_dilate_kernel": generation_outer_kernel,
        "generation_mask_dog_edge_kernel": generation_dog_edge_kernel,
        "dog_core_protect_kernel": dog_core_protect_kernel,
        "commit_mask_outer_dilate_kernel": commit_outer_kernel,
        "commit_mask_blur": int(COMMIT_MASK_BLUR),
        "mask_close_kernel": close_kernel,
        "dog_edge_no_diffuse_kernel": edge_no_diffuse_kernel,
        "dog_edge_soft_preserve_kernel": edge_soft_preserve_kernel,
        "diffusion_blend_blur": int(DIFFUSION_BLEND_BLUR),
        "pre_diffusion_blend_blur": int(PRE_DIFFUSION_BLEND_BLUR),
        "positive_prompt": DEFAULT_POSITIVE_PROMPT,
        "negative_prompt": DEFAULT_NEGATIVE_PROMPT,
        "outputs": {
            "seed_composite": str(output_dir / "seed_composite.png"),
            "luminance_harmonized_seed": str(output_dir / "luminance_harmonized_seed.png"),
            "pre_diffusion_seed": str(output_dir / "pre_diffusion_seed.png"),
            "route_seed": str(output_dir / "route_seed.png"),
            "masked_preview": str(output_dir / "masked_preview.png"),
            "refinement_mask": str(output_dir / "refinement_mask.png"),
            "route_generation_mask": str(output_dir / "route_generation_mask.png"),
            "route_refinement_mask": str(output_dir / "route_refinement_mask.png"),
            "effective_generation_mask": str(output_dir / "effective_generation_mask.png"),
            "effective_refinement_mask": str(output_dir / "effective_refinement_mask.png"),
            "generation_mask": str(output_dir / "generation_mask.png"),
            "commit_mask": str(output_dir / "commit_mask.png"),
            "dog_core_protect_mask": str(output_dir / "dog_core_protect_mask.png"),
            "dog_edge_generation_band": str(output_dir / "dog_edge_generation_band.png"),
            "base_refinement_mask": str(output_dir / "base_refinement_mask.png"),
            "residual_hole_mask": str(output_dir / "residual_hole_mask.png"),
            "expanded_residual_hole_mask": str(output_dir / "expanded_residual_hole_mask.png"),
            "outer_seam_only_mask": str(output_dir / "outer_seam_only_mask.png"),
            "protected_dog_zone": str(output_dir / "protected_dog_zone.png"),
            "dog_edge_no_diffuse_zone": str(output_dir / "dog_edge_no_diffuse_zone.png"),
            "dog_edge_soft_preserve_zone": str(output_dir / "dog_edge_soft_preserve_zone.png"),
            "outer_seam_for_diffusion": str(output_dir / "outer_seam_for_diffusion.png"),
            "pre_diffusion_blend_mask": str(output_dir / "pre_diffusion_blend_mask.png"),
            "contact_shadow_alpha": str(output_dir / "contact_shadow_alpha.png"),
            "refined_final": str(output_dir / "refined_final.png"),
            "zoom_seed": str(output_dir / "zoom_seed.png"),
            "zoom_refined": str(output_dir / "zoom_refined.png"),
        },
    }

    with open(output_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(run_config, f, ensure_ascii=False, indent=2)

    print("Saved refinement package to:", output_dir)
    print("Run config:", output_dir / "run_config.json")

    plot_images(
        [
            ("Pre-diffusion seed", pre_diffusion_seed),
            ("Refined final", refined_image),
            ("Zoom seed", zoom_seed),
            ("Zoom refined", zoom_refined),
        ],
        cols=2,
        figsize=(16, 12),
    )

    REFINE_RESULTS.append(
        {
            "pair_name": job["pair_name"],
            "pair_index": pair_index,
            "output_dir": str(output_dir),
            "run_config_path": str(output_dir / "run_config.json"),
            "used_model_id": used_model_id,
        }
    )

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(REFINE_RESULTS, f, ensure_ascii=False, indent=2)


## Export The Final Result And Presentation Assets

Each pair writes its own final refinement package under:

- `data/outputs/03_diffusion_refine/<pair_name>/`


In [ ]:
with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(REFINE_RESULTS, f, ensure_ascii=False, indent=2)

print("Processed refinement pairs:")
for item in REFINE_RESULTS:
    print(" -", item["pair_index"], item["pair_name"], "->", item["output_dir"])
print("Saved batch manifest:", OUTPUT_ROOT / "batch_manifest.json")


In [ ]:
pass
